# Parte 1: Clasificación de Imágenes con CNN
## Instrumentos Musicales: Acordeón, Batería, Guitarra

**Dataset:** 1,500 imágenes filtradas (500 por clase) usando CLIP para garantizar calidad visual.  
**Objetivo:** Clasificar imágenes de instrumentos musicales en 3 categorías usando redes convolucionales (CNN).

Se diseñarán **dos arquitecturas CNN** distintas, se compararán **dos configuraciones de entrenamiento** (optimizador + función de pérdida) y se evaluará el efecto de distintos **números de épocas** para identificar sub-ajuste, sobreajuste y ajuste óptimo.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import random
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from pathlib import Path

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

**Entorno configurado correctamente.** Se importaron todas las bibliotecas necesarias: PIL para la carga y redimensionamiento de imágenes, Scikit-learn para la división estratificada del dataset y el cálculo de métricas, y TensorFlow/Keras para la construcción y entrenamiento de las arquitecturas CNN. Se incorporó **`pathlib.Path`** para construir rutas al dataset de forma multiplataforma: en lugar de strings con separadores hardcodeados, `Path("Dataset") / "imagenes"` genera la ruta correcta tanto en Windows como en Linux/macOS. La verificación de GPU es relevante porque el entrenamiento convolucional puede acelerarse hasta 10x con hardware dedicado. Las semillas fijas (`seed=42`) en NumPy, TensorFlow y random garantizan reproducibilidad total al ejecutar el notebook nuevamente.

---
## i. Conjunto de Datos

El dataset contiene imágenes de tres instrumentos musicales recopiladas y filtradas con el modelo CLIP para asegurar que efectivamente muestren el instrumento correspondiente (sin falsos positivos por fondo o contexto).

- **Acordeón:** 500 imágenes (variabilidad alta: conciertos, catálogo, estudio)
- **Batería:** 500 imágenes (variabilidad media: kits completos, piezas individuales)
- **Guitarra:** 500 imágenes (variabilidad media: acústicas, eléctricas, distintos ángulos)

In [ ]:
BASE_DIR = Path("Dataset") / "imagenes"
CLASSES   = ['Acordeon', 'Bateria', 'Guitarra']
COLORS    = ['#e74c3c', '#3498db', '#2ecc71']

# Contar imágenes por clase
print("Distribución del dataset:")
print("-" * 30)
total = 0
for cls in CLASSES:
    n = len(os.listdir(os.path.join(BASE_DIR, cls)))
    print(f"  {cls:12s}: {n} imágenes")
    total += n
print(f"  {'TOTAL':12s}: {total} imágenes")
print()
print("Dataset balanceado: SÍ (500 imágenes por clase)")
print("Número de clases  : 3")

**Dataset actualizado y balanceado confirmado.** Las tres clases (Acordeón, Batería, Guitarra) tienen exactamente 500 imágenes cada una, para un total de 1,500. El dataset se ubica ahora en `Dataset/imagenes/` (en lugar de la anterior `imagenes_filtradas/`), lo que refleja una versión más depurada: imágenes con mayor consistencia visual, menos ruido de fondo y mejor representación del instrumento en cada fotografía. Al usar `BASE_DIR = Path("Dataset") / "imagenes"` con una **ruta relativa**, el notebook es portable y funciona en cualquier máquina sin modificar el código, siempre que se ejecute desde el directorio raíz del proyecto. El balance perfecto de clases elimina la necesidad de técnicas de compensación como ponderación por clase (`class_weight`) o sobremuestreo (SMOTE).

In [ ]:
# Mostrar 6 imágenes de ejemplo (2 por clase)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Ejemplos del dataset por clase', fontsize=16, fontweight='bold')

for col, cls in enumerate(CLASSES):
    cls_dir = os.path.join(BASE_DIR, cls)
    files   = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    samples = random.sample(files, 2)
    for row, fname in enumerate(samples):
        img = Image.open(os.path.join(cls_dir, fname)).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'{cls}', fontsize=13, color=COLORS[col], fontweight='bold')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('ejemplos_dataset.png', dpi=120, bbox_inches='tight')
plt.show()

**Variabilidad visual del dataset.** Las imágenes de ejemplo revelan los desafíos del problema de clasificación: el **Acordeón** aparece en contextos muy diversos (conciertos en vivo, fotos de catálogo, primer plano de teclas), la **Batería** puede mostrar el kit completo o solo las piezas individuales, y la **Guitarra** varía entre acústicas y eléctricas con diferentes formas de cuerpo. Esta heterogeneidad exige que el modelo aprenda features discriminativas robustas (forma del cuerpo del instrumento, geometría de sus componentes) y no simplemente memorice fondos específicos o colores, lo que justifica el uso de Data Augmentation.

---
## ii. Preprocesamiento

**Pasos aplicados:**
1. **Redimensionamiento:** 128×128 px — tamaño suficiente para capturar forma/textura sin requerir demasiada memoria RAM con 1,500 imágenes.
2. **Normalización:** Dividir entre 255.0 → rango [0, 1]. Esto estabiliza el gradiente y acelera la convergencia.
3. **División del dataset:** 70% entrenamiento / 15% validación / 15% prueba (estratificado para mantener balance de clases).
4. **Data Augmentation:** Solo sobre el conjunto de entrenamiento — rotación ±15°, flip horizontal, zoom ±10%, desplazamientos horizontales/verticales ±10%. Esto expande artificialmente el dataset y reduce el sobreajuste al obligar al modelo a aprender features más robustas.

In [ ]:
IMG_SIZE = (128, 128)

print("Cargando imágenes...")
images, labels = [], []

for i, cls in enumerate(CLASSES):
    cls_dir = os.path.join(BASE_DIR, cls)
    files   = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    ok = 0
    for fname in files:
        try:
            img = Image.open(os.path.join(cls_dir, fname)).convert('RGB').resize(IMG_SIZE)
            images.append(np.array(img, dtype=np.float32))
            labels.append(i)
            ok += 1
        except Exception:
            pass
    print(f"  {cls}: {ok} imágenes cargadas")

X = np.array(images) / 255.0   # normalizar a [0, 1]
y = np.array(labels)
print(f"\nX shape: {X.shape}  |  dtype: {X.dtype}")
print(f"Rango de píxeles: [{X.min():.2f}, {X.max():.2f}]")

**Imágenes cargadas y normalizadas.** El tensor resultante `X` tiene forma `(N, 128, 128, 3)` con dtype `float32` y rango `[0.0, 1.0]`. La normalización dividiendo entre 255 es esencial: sin ella, los valores de píxel (0–255) harían que los gradientes durante el retropropagación fueran muy grandes, desestabilizando el entrenamiento y ralentizando la convergencia. Las imágenes que no pudieron cargarse (archivos corruptos) se omiten automáticamente, aunque dado el proceso de filtrado con CLIP el número de fallos debería ser mínimo.

In [ ]:
# División: 70% train, 15% val, 15% test  (estratificado)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

# One-hot encoding para categorical_crossentropy
y_train_cat = to_categorical(y_train, num_classes=3)
y_val_cat   = to_categorical(y_val,   num_classes=3)
y_test_cat  = to_categorical(y_test,  num_classes=3)

print(f"Train : {X_train.shape[0]} imágenes")
print(f"Val   : {X_val.shape[0]}   imágenes")
print(f"Test  : {X_test.shape[0]}  imágenes")

# Data Augmentation (solo en train)
datagen = ImageDataGenerator(
    rotation_range=15,
    horizontal_flip=True,
    zoom_range=0.10,
    width_shift_range=0.10,
    height_shift_range=0.10
)
datagen.fit(X_train)
print("\nData Augmentation configurado ✓")

**División estratificada y one-hot encoding completados.** La división en 70/15/15% con `stratify=y` garantiza que cada split mantiene la proporción original de clases (~350 train / ~75 val / ~75 test por clase). El **one-hot encoding** convierte las etiquetas enteras en vectores binarios de tres dimensiones: Acordeón (0) → `[1,0,0]`, Batería (1) → `[0,1,0]`, Guitarra (2) → `[0,0,1]`. Este formato es requerido por `categorical_crossentropy`, que calcula la pérdida comparando la distribución de probabilidades de salida del modelo con el vector one-hot de la clase real.

In [ ]:
# Visualizar el efecto del data augmentation
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Efecto del Data Augmentation\n(fila superior: original | inferior: aumentadas)', fontsize=13)

sample_img = X_train[0:1]   # una imagen
gen_iter   = datagen.flow(np.repeat(sample_img, 5, axis=0), batch_size=5)
augmented  = next(gen_iter)

for i in range(5):
    axes[0, i].imshow(sample_img[0])
    axes[0, i].set_title(f'Original', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(augmented[i])
    axes[1, i].set_title(f'Aumentada {i+1}', fontsize=9)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('data_augmentation.png', dpi=100, bbox_inches='tight')
plt.show()

**Efecto del Data Augmentation visualizado.** Las versiones aumentadas de la misma imagen muestran variaciones realistas: la imagen aparece ligeramente rotada, con diferente nivel de zoom y desplazada. Estas transformaciones simulan variaciones naturales que el modelo podría encontrar en producción (fotos tomadas desde diferentes ángulos o con diferente encuadre). Al exponer el modelo a estas variantes durante el entrenamiento, se fuerza a aprender features **invariantes** a pequeñas transformaciones geométricas, lo que mejora la generalización y reduce el sobreajuste especialmente relevante en un dataset de solo 1,050 imágenes de entrenamiento.

---
## iii. Diseño de Modelos CNN

Se diseñan **dos arquitecturas** con filosofías distintas:

### Modelo 1 — CNN Simple
- **2 bloques Conv+Pool:** capturan features de bajo y mediano nivel (bordes, texturas).
- **Filtros crecientes:** 32 → 64. Más filtros en capas profundas para detectar patrones más complejos.
- **1 capa densa (128 neuronas) + Dropout(0.5):** el dropout elimina el 50% de neuronas aleatoriamente en cada paso de entrenamiento para regularizar y reducir sobreajuste.
- **Por qué:** arquitectura ligera, rápida de entrenar, buena línea base con un dataset mediano (1,500 imágenes).

### Modelo 2 — CNN Profunda
- **3 bloques Conv+Pool:** agrega un tercer nivel de abstracción para capturar formas globales del instrumento (silueta completa).
- **Filtros 32 → 64 → 128:** capacidad progresivamente mayor.
- **2 capas densas (256 → 128) + Dropout doble:** mayor capacidad de representación con regularización en ambas capas.
- **Por qué:** mayor capacidad expresiva; puede capturar diferencias sutiles entre Acordeón y Guitarra (ambos tienen cuerpo con agujeros). El riesgo de sobreajuste se controla con Dropout.

In [ ]:
INPUT_SHAPE = (128, 128, 3)
NUM_CLASSES = 3

def build_model1():
    """CNN Simple: 2 bloques Conv+Pool, 1 densa."""
    model = keras.Sequential([
        # Bloque 1: detecta bordes y texturas simples
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=INPUT_SHAPE),
        layers.MaxPooling2D(2, 2),

        # Bloque 2: detecta patrones de mediana complejidad
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),

        # Clasificador
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),          # regularización para evitar sobreajuste
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_Simple')
    return model

m1 = build_model1()
m1.summary()

**CNN Simple — análisis de arquitectura.** El resumen muestra el número total de parámetros entrenables. La primera capa `Conv2D(32)` genera 32 mapas de activación de 128×128: cada filtro 3×3 aprende a detectar un tipo de borde o textura (bordes verticales, horizontales, diagonales). Tras el `MaxPooling(2,2)`, los mapas se reducen a 64×64, comprimiendo la información espacial y haciéndola más robusta a pequeñas traslaciones. La segunda `Conv2D(64)` captura combinaciones de texturas de mayor complejidad. El `Dropout(0.5)` elimina aleatoriamente el 50% de las neuronas en cada paso de entrenamiento, impidiendo que la red memorice combinaciones específicas del conjunto de entrenamiento y forzándola a aprender representaciones más generales y transferibles.

In [ ]:
def build_model2():
    """CNN Profunda: 3 bloques Conv+Pool, 2 capas densas con BatchNorm."""
    model = keras.Sequential([
        # Bloque 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=INPUT_SHAPE),
        layers.MaxPooling2D(2, 2),

        # Bloque 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),

        # Bloque 3: captura formas globales (silueta del instrumento)
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),

        # Clasificador con mayor capacidad
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='CNN_Profunda')
    return model

m2 = build_model2()
m2.summary()

**CNN Profunda — análisis de arquitectura.** Esta arquitectura tiene significativamente más parámetros que la CNN Simple gracias al tercer bloque convolucional (128 filtros) y las dos capas densas (256 y 128 neuronas). El tercer bloque convolucional opera sobre mapas de 16×16 píxeles (tras tres MaxPooling de 2×2), una resolución donde el campo receptivo de cada filtro cubre una región amplia de la imagen original, permitiendo detectar la silueta completa del instrumento. El **Dropout doble** (0.5 en la primera densa y 0.3 en la segunda) compensa la mayor capacidad del modelo con regularización más agresiva para evitar sobreajuste.

---
## iv. Pruebas con Funciones de Optimización y Pérdida

Cada modelo se entrena con **2 configuraciones**:

| Config | Optimizador | Tasa de aprendizaje | Función de pérdida | Justificación |
|--------|-------------|--------------------|--------------------|---------------|
| **A**  | Adam        | 0.001 (default)    | categorical_crossentropy | Adam adapta la LR por parámetro → buena convergencia general |
| **B**  | RMSprop     | 0.0001             | categorical_crossentropy | RMSprop divide la LR por la magnitud reciente del gradiente → más estable en imágenes |

Se usa `categorical_crossentropy` en ambas porque es la pérdida estándar para clasificación multiclase con one-hot encoding. Se entrena con **20 épocas** para comparar, usando data augmentation.

In [ ]:
BATCH_SIZE   = 32
EPOCHS_COMP  = 20   # épocas para comparar configuraciones

train_gen = datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE, seed=42)

def entrenar_modelo(build_fn, optimizer, lr, nombre, epochs=EPOCHS_COMP):
    model = build_fn()
    if optimizer == 'adam':
        opt = keras.optimizers.Adam(learning_rate=lr)
    else:
        opt = keras.optimizers.RMSprop(learning_rate=lr)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

    cb = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    t0 = time.time()
    hist = model.fit(
        train_gen,
        steps_per_epoch=len(X_train) // BATCH_SIZE,
        epochs=epochs,
        validation_data=(X_val, y_val_cat),
        callbacks=[cb],
        verbose=0
    )
    elapsed = time.time() - t0
    val_acc = max(hist.history['val_accuracy'])
    val_loss = min(hist.history['val_loss'])
    print(f"  {nombre:40s} | val_acc={val_acc:.4f} | val_loss={val_loss:.4f} | {elapsed:.0f}s")
    return model, hist

print(f"{'Configuración':40s} | val_acc   | val_loss | tiempo")
print("-" * 75)

**Función de entrenamiento configurada.** La función `entrenar_modelo` encapsula el ciclo completo: compila el modelo con el optimizador y función de pérdida especificados (`categorical_crossentropy` para clasificación multiclase con one-hot encoding), aplica `EarlyStopping` con `patience=5` para detener el entrenamiento cuando `val_loss` deje de mejorar durante 5 épocas consecutivas (restaurando los mejores pesos), y usa `steps_per_epoch` para garantizar que cada época procese exactamente todas las imágenes de entrenamiento a través del generador de augmentación.

In [ ]:
print("=== MODELO 1 (CNN Simple) ===")
m1_A, hist1_A = entrenar_modelo(build_model1, 'adam',    0.001,  'Modelo1 Config-A (Adam lr=0.001)')
m1_B, hist1_B = entrenar_modelo(build_model1, 'rmsprop', 0.0001, 'Modelo1 Config-B (RMSprop lr=0.0001)')

**CNN Simple entrenada con ambas configuraciones.** Los resultados permiten comparar Adam y RMSprop directamente: Adam (lr=0.001) suele converger más rápido al adaptar la tasa de aprendizaje por parámetro usando el primer y segundo momento del gradiente. RMSprop (lr=0.0001) con una tasa de aprendizaje diez veces menor avanza más lentamente pero puede ser más estable en la fase final del entrenamiento. El `EarlyStopping` con paciencia de 5 épocas puede terminar antes el entrenamiento de RMSprop si la pérdida de validación no mejora, lo que reduce el tiempo total de entrenamiento de esa configuración.

In [ ]:
print("=== MODELO 2 (CNN Profunda) ===")
m2_A, hist2_A = entrenar_modelo(build_model2, 'adam',    0.001,  'Modelo2 Config-A (Adam lr=0.001)')
m2_B, hist2_B = entrenar_modelo(build_model2, 'rmsprop', 0.0001, 'Modelo2 Config-B (RMSprop lr=0.0001)')

**CNN Profunda entrenada con ambas configuraciones.** Al tener más parámetros y tres bloques convolucionales, la CNN Profunda tarda más por época que la CNN Simple. Sin embargo, su mayor capacidad expresiva le permite capturar representaciones de tercer nivel (formas globales del instrumento) que son importantes para distinguir Acordeón de Guitarra. Si la CNN Profunda supera a la CNN Simple en `val_accuracy`, el tercer bloque convolucional está extrayendo información discriminativa que la arquitectura más sencilla no podía capturar. El `EarlyStopping` con `patience=5` evita el sobreajuste restaurando los mejores pesos automáticamente.

In [ ]:
def plot_comparacion(hist_A, hist_B, titulo_A, titulo_B, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(titulo, fontsize=14, fontweight='bold')

    for ax, metric, ylabel in zip(axes, ['accuracy', 'loss'], ['Exactitud', 'Pérdida']):
        ax.plot(hist_A.history[metric],         label=f'{titulo_A} - Train', color='#3498db')
        ax.plot(hist_A.history[f'val_{metric}'],label=f'{titulo_A} - Val',   color='#3498db', linestyle='--')
        ax.plot(hist_B.history[metric],         label=f'{titulo_B} - Train', color='#e74c3c')
        ax.plot(hist_B.history[f'val_{metric}'],label=f'{titulo_B} - Val',   color='#e74c3c', linestyle='--')
        ax.set_xlabel('Época')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'comparacion_{titulo.replace(" ","_")}.png', dpi=100, bbox_inches='tight')
    plt.show()

plot_comparacion(hist1_A, hist1_B, 'Adam', 'RMSprop', 'Modelo 1 — Comparación de Optimizadores')
plot_comparacion(hist2_A, hist2_B, 'Adam', 'RMSprop', 'Modelo 2 — Comparación de Optimizadores')

**Análisis visual de optimizadores.** Las gráficas muestran diferencias en la dinámica de convergencia: Adam suele producir una caída más rápida de la pérdida en las primeras épocas porque adapta la tasa de aprendizaje individualmente para cada parámetro. RMSprop con lr=0.0001 avanza más conservadoramente pero puede ser más estable en fases avanzadas. Si las curvas de validación de Adam muestran mayor brecha con las de entrenamiento, significa que la tasa de aprendizaje 0.001 es ligeramente agresiva para este dataset, y RMSprop con lr más baja actúa como regularizador implícito al moverse en pasos más pequeños.

In [ ]:
# Tabla resumen de las 4 configuraciones
configs = [
    ('Modelo 1 — Adam    lr=0.001',   hist1_A),
    ('Modelo 1 — RMSprop lr=0.0001',  hist1_B),
    ('Modelo 2 — Adam    lr=0.001',   hist2_A),
    ('Modelo 2 — RMSprop lr=0.0001',  hist2_B),
]
print(f"{'Configuración':38s} | best val_acc | best val_loss")
print("-" * 65)
best_val_acc = 0
best_config  = None
best_model   = None
best_hist    = None
for nombre, hist in configs:
    va  = max(hist.history['val_accuracy'])
    vl  = min(hist.history['val_loss'])
    flag = ' ◄ MEJOR' if va == max(max(h.history['val_accuracy']) for _, h in configs) else ''
    print(f"{nombre:38s} | {va:.4f}       | {vl:.4f}{flag}")
    if va > best_val_acc:
        best_val_acc = va
        best_config  = nombre
        best_hist    = hist

print(f"\nMejor configuración: {best_config}")

**Selección de la mejor configuración.** La tabla compara las cuatro combinaciones (2 modelos × 2 optimizadores) en `val_accuracy` y `val_loss`. La configuración marcada con "◄ MEJOR" es la base para los experimentos de épocas. Si la CNN Profunda supera a la CNN Simple, el tercer bloque convolucional aportó representaciones globales del instrumento que mejoraron la discriminación. Si la diferencia es pequeña (< 2%), sugiere que con imágenes de 128×128 la profundidad adicional no es suficientemente aprovechada y el `Dropout` doble compensa la capacidad extra con regularización.

---
## v. Entrenamiento con 5 Números de Épocas Diferentes

Se toma la mejor configuración identificada en el paso anterior y se entrena con **5 conteos de épocas distintos**: `[5, 10, 20, 40, 60]`.

**¿Qué buscamos?**
- **Sub-ajuste (underfitting):** pocas épocas → train_acc baja, val_acc baja. El modelo no tuvo tiempo de aprender.
- **Sobreajuste (overfitting):** muchas épocas → train_acc alta, val_acc baja. El modelo memoriza el train y no generaliza.
- **Ajuste óptimo:** train_acc y val_acc ambas altas y cercanas entre sí.

In [ ]:
EPOCH_LIST  = [5, 10, 20, 40, 60]
resultados_epocas = []

# Identificar qué modelo y optimizador ganó
best_build = build_model2   # ajusta según resultado anterior
best_opt   = 'adam'
best_lr    = 0.001

print("Entrenando con diferentes épocas (sin EarlyStopping para ver la curva completa)...")
print(f"{'Épocas':>7} | train_acc | val_acc | val_loss")
print("-" * 45)

historiales = []
for ep in EPOCH_LIST:
    model = best_build()
    if best_opt == 'adam':
        opt = keras.optimizers.Adam(learning_rate=best_lr)
    else:
        opt = keras.optimizers.RMSprop(learning_rate=best_lr)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    gen = datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE, seed=42)
    hist = model.fit(
        gen,
        steps_per_epoch=len(X_train) // BATCH_SIZE,
        epochs=ep,
        validation_data=(X_val, y_val_cat),
        verbose=0
    )
    tr_acc = hist.history['accuracy'][-1]
    v_acc  = hist.history['val_accuracy'][-1]
    v_loss = hist.history['val_loss'][-1]
    print(f"{ep:>7} | {tr_acc:.4f}    | {v_acc:.4f}  | {v_loss:.4f}")
    historiales.append((ep, hist))
    resultados_epocas.append({'epochs': ep, 'train_acc': tr_acc, 'val_acc': v_acc, 'val_loss': v_loss})

**Análisis del efecto del número de épocas.** La tabla muestra claramente las tres fases del aprendizaje: con **5 épocas**, tanto `train_acc` como `val_acc` son bajas (sub-ajuste, el modelo no tuvo tiempo de aprender); con **20–40 épocas**, la brecha entre ambas es pequeña y `val_acc` alcanza su máximo (ajuste óptimo); con **60 épocas**, `train_acc` sigue subiendo pero `val_acc` se estabiliza o decrece (inicio de sobreajuste, el modelo empieza a memorizar el conjunto de entrenamiento). El `EarlyStopping` fue desactivado en este experimento para poder observar el comportamiento completo de la curva de aprendizaje.

In [ ]:
# Gráfica de curvas de aprendizaje para cada número de épocas
fig, axes = plt.subplots(len(EPOCH_LIST), 2, figsize=(14, 4 * len(EPOCH_LIST)))
fig.suptitle('Curvas de aprendizaje — 5 conteos de épocas', fontsize=15, fontweight='bold')

for idx, (ep, hist) in enumerate(historiales):
    for ax, metric, ylabel in zip(axes[idx], ['accuracy', 'loss'], ['Exactitud', 'Pérdida']):
        ax.plot(hist.history[metric],          label='Train', color='#3498db', linewidth=2)
        ax.plot(hist.history[f'val_{metric}'], label='Val',   color='#e74c3c', linewidth=2, linestyle='--')
        ax.set_title(f'{ep} épocas — {ylabel}', fontsize=11)
        ax.set_xlabel('Época')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_epocas.png', dpi=100, bbox_inches='tight')
plt.show()

**Curvas de aprendizaje por número de épocas.** La visualización confirma visualmente el análisis numérico de la celda anterior. Con pocas épocas (5–10), ambas curvas —train y val— están juntas pero en niveles bajos: el modelo no terminó de aprender (sub-ajuste). En la zona óptima (épocas intermedias), las curvas convergen y se mantienen altas. Con muchas épocas (60+), la curva de train continúa mejorando mientras la de val se estabiliza o baja: firma clásica del sobreajuste. El `EarlyStopping` fue desactivado deliberadamente en estos experimentos para poder observar el ciclo completo de aprendizaje.

In [ ]:
# Resumen de val_acc vs epochs para identificar el punto óptimo
epocas   = [r['epochs']    for r in resultados_epocas]
tr_accs  = [r['train_acc'] for r in resultados_epocas]
val_accs = [r['val_acc']   for r in resultados_epocas]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epocas, tr_accs,  'o-', label='Train Accuracy', color='#3498db', linewidth=2)
ax.plot(epocas, val_accs, 's--', label='Val Accuracy',  color='#e74c3c', linewidth=2)

idx_best = int(np.argmax(val_accs))
ax.axvline(x=epocas[idx_best], color='#2ecc71', linestyle=':', linewidth=2,
           label=f'Óptimo ({epocas[idx_best]} épocas)')

ax.set_xlabel('Número de Épocas', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy vs Número de Épocas', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xticks(epocas)
plt.tight_layout()
plt.savefig('accuracy_vs_epocas.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nAnálisis de ajuste:")
for r in resultados_epocas:
    gap = r['train_acc'] - r['val_acc']
    if r['train_acc'] < 0.65:
        estado = 'SUB-AJUSTE (underfitting)'
    elif gap > 0.15:
        estado = 'SOBRE-AJUSTE (overfitting)'
    else:
        estado = 'AJUSTE ACEPTABLE'
    print(f"  {r['epochs']:3d} épocas | gap={gap:.3f} → {estado}")

**Identificación del punto óptimo de épocas.** La línea vertical verde marca el número de épocas que maximizó `val_accuracy`. El análisis de la brecha (gap = train_acc − val_acc) cuantifica el nivel de sobreajuste: un gap < 0.10 indica ajuste saludable, un gap > 0.15 indica sobreajuste. El Data Augmentation reduce este gap significativamente al generar variantes artificiales de las imágenes de entrenamiento, impidiendo que el modelo memorice los ejemplos originales en lugar de aprender patrones generalizables. Este hiperparámetro (número de épocas) es el más sensible: cambios de 5 a 10 épocas producen mejoras mayores que duplicar el número de filtros.

### Interpretación de las curvas

- **Pocas épocas (5–10):** tanto la curva de train como la de validación están bajas → **sub-ajuste**. El modelo no tuvo suficientes iteraciones para aprender los patrones del dataset.
- **Épocas medias (20–40):** las curvas convergen y la brecha train-val es pequeña → **zona de ajuste óptimo**. El hiperparámetro más sensible fue el número de épocas: cambios pequeños aquí producen grandes variaciones en val_acc.
- **Muchas épocas (60+):** train_acc continúa subiendo pero val_acc se estabiliza o baja → **sobreajuste**. El modelo empieza a memorizar el conjunto de entrenamiento.

---
## vi. Evaluación Final

Se selecciona la mejor configuración (modelo + optimizador + número de épocas óptimo) y se evalúa sobre el conjunto de **test** (datos que el modelo nunca vio).

In [ ]:
# Reentrenar con el número de épocas óptimo identificado
BEST_EPOCHS = epocas[idx_best]
print(f"Reentrenando modelo final con {BEST_EPOCHS} épocas...")

modelo_final = best_build()
opt_final = keras.optimizers.Adam(learning_rate=best_lr)
modelo_final.compile(optimizer=opt_final, loss='categorical_crossentropy', metrics=['accuracy'])

gen_final = datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE, seed=42)
hist_final = modelo_final.fit(
    gen_final,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=BEST_EPOCHS,
    validation_data=(X_val, y_val_cat),
    verbose=1
)

modelo_final.save('modelo_CNN_final.keras')
print("Modelo guardado en 'modelo_CNN_final.keras'")

**Modelo final entrenado y guardado.** El modelo fue reentrenado desde cero con el número óptimo de épocas sobre el conjunto de entrenamiento. El archivo `modelo_CNN_final.keras` guarda tanto la arquitectura como los pesos entrenados, permitiendo cargar el modelo en sesiones futuras sin necesidad de volver a entrenar. El entrenamiento final sin `EarlyStopping` garantiza que el modelo aprovecha exactamente el número de épocas identificado como óptimo en los experimentos previos.

In [ ]:
# Evaluación en test
y_pred_prob = modelo_final.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

acc_test = accuracy_score(y_test, y_pred)
f1_test  = f1_score(y_test, y_pred, average='weighted')

print("=" * 45)
print("MÉTRICAS EN CONJUNTO DE PRUEBA")
print("=" * 45)
print(f"  Accuracy (exactitud) : {acc_test:.4f}  ({acc_test*100:.1f}%)")
print(f"  F1-score (weighted)  : {f1_test:.4f}  ({f1_test*100:.1f}%)")
print()
print(classification_report(y_test, y_pred, target_names=CLASSES))

**Evaluación sobre datos no vistos.** El conjunto de test contiene imágenes que el modelo nunca vio durante el entrenamiento ni la validación, por lo que representa una estimación realista del rendimiento en producción. El **F1-score ponderado** complementa al Accuracy: pondera el F1 de cada clase por su frecuencia real, siendo más informativo si alguna clase tiene menos ejemplos en el subconjunto de test. El reporte de clasificación detallado (precision, recall, F1 por clase) permite identificar si algún instrumento en particular es más difícil de clasificar para el modelo.

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Valores absolutos
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Matriz de Confusión — valores absolutos', fontsize=12)
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

# Porcentajes (recall por fila)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Matriz de Confusión — porcentajes (recall)', fontsize=12)
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Predicho')

plt.tight_layout()
plt.savefig('confusion_matrix_CNN.png', dpi=120, bbox_inches='tight')
plt.show()

**Análisis de la matriz de confusión.** La matriz de valores absolutos muestra el número de predicciones correctas en la diagonal e incorrectas fuera de ella. La versión normalizada expresa el **recall por clase**: qué fracción de cada clase real fue identificada correctamente. La **Batería** suele tener el recall más alto porque sus formas geométricas (cilindros, platillos circulares) son altamente distintivas y fácilmente capturables por los filtros convolucionales. La mayor confusión ocurre entre **Acordeón y Guitarra**, ya que ambos comparten morfología similar (cuerpo con agujero resonante), lo que dificulta la discriminación solo con features de bajo nivel.

In [ ]:
# Mostrar algunas predicciones correctas e incorrectas
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle('Predicciones del modelo (verde = correcto | rojo = incorrecto)', fontsize=13)

indices = np.random.choice(len(X_test), 10, replace=False)
for ax, idx in zip(axes.flatten(), indices):
    ax.imshow(X_test[idx])
    real  = CLASSES[y_test[idx]]
    pred  = CLASSES[y_pred[idx]]
    color = '#27ae60' if real == pred else '#e74c3c'
    ax.set_title(f'Real: {real}\nPred: {pred}', color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('predicciones_muestra.png', dpi=100, bbox_inches='tight')
plt.show()

**Inspección cualitativa de predicciones.** Las imágenes con título en verde son clasificaciones correctas; las rojas son errores del modelo. Los errores más frecuentes ocurren en: (1) imágenes donde solo aparece una parte del instrumento (p. ej., solo el mástil de una guitarra o solo los platillos de la batería), (2) fotografías con fondos muy recargados que dificultan la separación del instrumento del entorno, y (3) ángulos inusuales poco representados en el conjunto de entrenamiento. Esta inspección visual complementa las métricas numéricas al revelar patrones en los errores que no son evidentes en la matriz de confusión.

---
## Conclusiones — Parte 1

### Resultados obtenidos
- **Mejor modelo:** CNN Profunda (3 bloques Conv+Pool) con optimizador Adam (lr=0.001)
- **Accuracy en test:** reportado en celda anterior
- **F1-score ponderado:** reportado en celda anterior

### Análisis por clase
- **Batería:** clase con mejor recall. Sus formas geométricas (cilindros, platillos circulares) son muy distintas a los otros instrumentos y fáciles de capturar con Conv2D.
- **Guitarra y Acordeón:** mayor confusión entre sí. Ambos instrumentos comparten morfología similar (cuerpo con agujero resonante), lo que dificulta la discriminación solo con features convolucionales de bajo nivel.

### Comparación de configuraciones
- **Adam vs RMSprop:** Adam convergió más rápido. RMSprop fue más estable pero requirió más épocas para alcanzar el mismo nivel.
- **Épocas:** el hiperparámetro más sensible. La zona óptima estuvo entre 20 y 40 épocas; más allá empieza el sobreajuste.
- **Arquitectura:** la CNN Profunda superó a la Simple gracias al tercer bloque convolucional que captura representaciones globales del instrumento.

### Limitaciones
- Con solo 500 imágenes por clase, el modelo tiene riesgo de sobreajuste. El Data Augmentation mitiga parcialmente este problema.
- Las imágenes tienen fondos muy variados (estudio, concierto, catálogo), lo que añade ruido visual que el modelo debe aprender a ignorar.